<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB16_Autoencoders_and_Anomaly_Detection_Real_Fuel_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB16 · Class 16 — Autoencoders and Anomaly Detection**

## Block 3: AI — Deep Learning (continued)

Every network so far in this block was **supervised**: `a known label (mine/rock, fuel type, next month's consumption) told the network what to predict`. This class returns to **unsupervised learning** (`NB09`'s territory) with a Deep Learning tool built for it: the **autoencoder**. We use it for one of its most common real applications — **anomaly detection** — applied once more to the real `ship_fuel_efficiency.csv` data from `NB07`/`NB09`/`NB14`, this time asking a new question: which voyages look *unusual*, and does "unusual" line up with anything we can actually explain?

### Learning objectives

By the end of this class, students will be able to:
- Explain what an autoencoder learns, and why forcing data through a narrow bottleneck is the key idea.
- Explain how reconstruction error becomes an anomaly score, with no labels required.
- Train a real autoencoder on real naval data and choose a defensible anomaly threshold.
- Interpret flagged anomalies against known-but-unused categorical fields, the same validation habit from `NB09`'s clustering class.
- Relate autoencoders to PCA (`NB09`) as two different routes to the same underlying idea: compression.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | Why unsupervised Deep Learning? Autoencoders as compression | 10 min | Theory |
| 3 | Autoencoder architecture: encoder, bottleneck, decoder | 10 min | Theory + Practice |
| 4 | From reconstruction error to an anomaly score | 10 min | Theory |
| 5 | Real dataset: revisiting ship fuel data with a new question | 10 min | Practice |
| 6 | Hands-on: building and training the autoencoder | 30 min | Practice |
| 7 | Choosing a threshold and flagging anomalies | 20 min | Practice |
| 8 | Interpreting anomalies against known fields | 10 min | Practice |
| 9 | Autoencoders vs. PCA, and beyond anomaly detection | 10 min | Theory + Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB11`–`NB13`**: supervised classifiers — tabular, then images.
- **`NB14`**: sequence forecasting with a real LSTM.
- **`NB15`**: transfer learning, feature extraction vs. fine-tuning.
- **`NB16`** (today): unsupervised Deep Learning — autoencoders, applied to anomaly detection.

One session remains after this to close Block 3 (`NB17`).

---

## 2. Why unsupervised Deep Learning? Autoencoders as compression

`NB09` did unsupervised learning without Deep Learning: K-Means found groups, PCA found lower-dimensional projections, and both worked entirely from the data's own structure, no labels needed. An **[autoencoder](https://en.wikipedia.org/wiki/Autoencoder)** applies that same "no labels" idea to a neural network: it's trained to reconstruct its own input as closely as possible, after squeezing it through a deliberately narrow **bottleneck** layer.

The trick is exactly that bottleneck. If the network could simply copy input to output, `it would need no bottleneck at all and would learn nothing useful` — a wide-open pipe copies anything. Forcing the data through a smaller layer means the network can only reconstruct well if it learns to **compress** the input into a compact representation that still captures whatever pattern is common across most of the data. Anything the network hasn't learned to represent well in that bottleneck — because it's rare, unusual, or simply different from most of the training examples — gets reconstructed *poorly*.

---

## 3. Autoencoder architecture: encoder, bottleneck, decoder

An autoencoder has two halves sharing one narrow middle layer:

- The **encoder** compresses the input down to the bottleneck.
- The **bottleneck** is the compressed representation — fewer numbers than the original input.
- The **decoder** expands the bottleneck back up to the original input's shape, attempting to reconstruct it.

Structurally, it's just two small MLPs (`NB11` territory) glued together, trained end to end to minimize **reconstruction error** — typically Mean Squared Error between input and output, `NB07`'s regression metric repurposed for a completely different goal:

Let's draw that hourglass shape:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

layer_sizes = [5, 3, 2, 3, 5]
layer_x = [0, 1.6, 3.2, 4.8, 6.4]
labels = ["Input\n(features)", "Encoder\nhidden", "Bottleneck\n(compressed)", "Decoder\nhidden", "Output\n(reconstruction)"]
colors = ["lightblue", "lightcoral", "gold", "lightcoral", "lightblue"]

positions = []
for n, x in zip(layer_sizes, layer_x):
    ys = np.linspace(-(n - 1) / 2, (n - 1) / 2, n)
    positions.append([(x, y) for y in ys])

fig, ax = plt.subplots(figsize=(10, 5))

for l in range(len(layer_sizes) - 1):
    for x1, y1 in positions[l]:
        for x2, y2 in positions[l + 1]:
            ax.plot([x1, x2], [y1, y2], color="lightgray", lw=0.7, zorder=1)

for layer, color, label, x in zip(positions, colors, labels, layer_x):
    for lx, ly in layer:
        ax.add_patch(patches.Circle((lx, ly), 0.15, facecolor=color, edgecolor="black", zorder=2))
    ax.text(x, -3, label, ha="center", fontsize=9)

ax.set_xlim(-1, 7.4)
ax.set_ylim(-3.6, 3)
ax.axis("off")
ax.set_title("Autoencoder: encoder -> bottleneck -> decoder")
plt.tight_layout()
plt.show()

The narrowest point (gold, 2 units wide here) is where compression is forced to happen. Note there is no separate "target" in the loss function — the input *is* `the target, which is exactly what makes this unsupervised`: no `y` column anywhere in today's training loop.

---

## 4. From reconstruction error to an anomaly score

Train the autoencoder on data that is **mostly normal** (a reasonable assumption for most operational data — most voyages are unremarkable; genuinely anomalous ones are rare by definition). The network's bottleneck learns to represent whatever pattern is common across most examples. Afterward, run *any* example through the trained network and measure how far the reconstruction is from the original — the **reconstruction error**:

$$
\text{error}(x) = \frac{1}{n} \lVert x - \text{decoder}(\text{encoder}(x)) \rVert^2
$$

A **typical** example reconstructs well (low error) — the network has effectively seen many similar examples and learned to represent that pattern. An **unusual** example reconstructs poorly (high error) — it doesn't fit the compressed representation the network learned, precisely because `the network never had to learn to represent rare patterns well to minimize its average training loss`.

This gives us anomaly detection with **zero labeled anomalies required** — a genuine practical advantage over the supervised classifiers from `NB08`/`NB11`–`NB13`, which all needed a labeled example of *every* class they could recognize. An autoencoder only ever needs examples of "normal."

---

## 5. Real dataset: revisiting ship fuel data with a new question

Same real `ship_fuel_efficiency.csv` as `NB07`/`NB09`/`NB14` — 1,440 real voyages, 120 ships. `NB07` asked "can we predict fuel consumption?"; `NB09` asked "do voyages cluster into operational profiles?"; today asks a third, genuinely different question: **which individual voyages look operationally unusual, using no labels at all?**

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
fuel.head()

Same leakage-safe numeric features as `NB07`/`NB09` — `distance`, `fuel_consumption`, `engine_efficiency` — still excluding `CO2_emissions` for the same redundancy reason established back in `NB07`:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import torch

feature_cols = ["distance", "fuel_consumption", "engine_efficiency"]
X = fuel[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_full_t = torch.tensor(X_scaled, dtype=torch.float32)  # every voyage, for scoring later

X_train_t.shape

---

## 6. Hands-on: building and training the autoencoder

3 real features compressed down to a 2-unit bottleneck — enough to force real compression, small enough to train in seconds:

In [ ]:
import torch.nn as nn

class FuelAutoencoder(nn.Module):
    def __init__(self, n_features, bottleneck_size=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 8),
            nn.ReLU(),
            nn.Linear(8, bottleneck_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 8),
            nn.ReLU(),
            nn.Linear(8, n_features),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

torch.manual_seed(42)
autoencoder = FuelAutoencoder(n_features=X_train_t.shape[1])
sum(p.numel() for p in autoencoder.parameters())

Train exactly like every network since `NB11` — except notice the loss compares the model's output to **its own input**, not to a separate label:

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.01)

n_epochs = 100
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    autoencoder.train()
    optimizer.zero_grad()
    reconstruction = autoencoder(X_train_t)
    loss = criterion(reconstruction, X_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    autoencoder.eval()
    with torch.no_grad():
        val_reconstruction = autoencoder(X_val_t)
        val_loss = criterion(val_reconstruction, X_val_t)
    val_losses.append(val_loss.item())

plt.plot(train_losses, label="Training reconstruction loss")
plt.plot(val_losses, label="Validation reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Autoencoder training")
plt.legend()
plt.show()

**Try it yourself**: seeing the loss curve fall is one thing — look at what the network actually reconstructs for a few real voyages, in real units (undoing the scaling), to see what "close to the original" concretely looks like.

In [ ]:
sample_idx = [0, 1, 2, 3, 4]
with torch.no_grad():
    reconstructed_sample = autoencoder(X_full_t[sample_idx]).numpy()

original_real = scaler.inverse_transform(X_full_t[sample_idx].numpy())
reconstructed_real = scaler.inverse_transform(reconstructed_sample)

comparison_df = pd.DataFrame({
    "distance (actual)": original_real[:, 0],
    "distance (reconstructed)": reconstructed_real[:, 0],
    "fuel_consumption (actual)": original_real[:, 1],
    "fuel_consumption (reconstructed)": reconstructed_real[:, 1],
})
comparison_df.round(1)


---

## 7. Choosing a threshold and flagging anomalies

Score **every** voyage (not just the held-out validation set) by its individual reconstruction error:

In [ ]:
autoencoder.eval()
with torch.no_grad():
    full_reconstruction = autoencoder(X_full_t)
    reconstruction_error = ((full_reconstruction - X_full_t) ** 2).mean(dim=1).numpy()

fuel["reconstruction_error"] = reconstruction_error
fuel["reconstruction_error"].describe()

There's no universally "correct" threshold — it's a policy choice, trading off how many voyages get flagged for review against how many genuine anomalies you're willing to miss (the same false-positive/false-negative tradeoff `NB13` §9 discussed, now without labels to measure it precisely against). A common, defensible starting point: flag the top few percent by error:

In [ ]:
threshold = np.percentile(reconstruction_error, 95)
fuel["is_anomaly"] = fuel["reconstruction_error"] > threshold

print(f"Threshold (95th percentile): {threshold:.3f}")
print(f"Flagged as anomalous: {fuel['is_anomaly'].sum()} / {len(fuel)} voyages")

Visualize the error distribution and where the threshold falls:

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(reconstruction_error, bins=40)
plt.axvline(threshold, color="red", linestyle="--", label="95th percentile threshold")
plt.xlabel("Reconstruction error")
plt.ylabel("Number of voyages")
plt.title("Reconstruction error distribution")
plt.legend()
plt.show()

**Try it yourself**: the reconstruction error above averages across all 3 features — break it down per feature for the worst-reconstructed voyages. Is one feature usually responsible, or do all three contribute roughly equally?

In [ ]:
with torch.no_grad():
    per_feature_error = ((full_reconstruction - X_full_t) ** 2).numpy()

top_anomaly_idx = np.argsort(reconstruction_error)[-5:][::-1]
pd.DataFrame(per_feature_error[top_anomaly_idx], columns=feature_cols).round(3)


---

## 8. Interpreting anomalies against known fields

Exactly the sanity-check habit from `NB09`'s clustering class: we flagged anomalies using **only** the 3 numeric features, never touching `weather_conditions` or `ship_type`. Do the flagged voyages line up with either, or did the autoencoder find something else entirely?

In [ ]:
pd.crosstab(fuel["is_anomaly"], fuel["weather_conditions"])

And by ship type:

In [ ]:
pd.crosstab(fuel["is_anomaly"], fuel["ship_type"])

**Read your own tables**: if anomalies concentrate in `Stormy` weather, that's a reassuring, physically sensible result — unusual conditions produce unusual fuel behavior, and the autoencoder found that without ever being told about weather. If anomalies are spread evenly across weather and ship type instead, that's equally informative — it suggests the flagged voyages are unusual for some other reason (a specific route, a maintenance issue, a data-entry error) worth investigating individually, not a pattern this crosstab alone can explain. A scatter plot adds a visual gut-check:

In [ ]:
plt.figure(figsize=(7, 5))
normal = fuel[~fuel["is_anomaly"]]
anomalous = fuel[fuel["is_anomaly"]]
plt.scatter(normal["distance"], normal["fuel_consumption"], alpha=0.4, label="Normal")
plt.scatter(anomalous["distance"], anomalous["fuel_consumption"], color="red", label="Flagged anomaly")
plt.xlabel("Distance (nm)")
plt.ylabel("Fuel consumption (L)")
plt.title("Flagged anomalies in distance/fuel-consumption space")
plt.legend()
plt.show()

**Try it yourself**: crosstabs and scatter plots show patterns in aggregate — look at the actual top 5 most anomalous voyages directly. Do their real feature values explain, in plain terms, why the autoencoder struggled to reconstruct them?

In [ ]:
top5 = fuel.iloc[top_anomaly_idx][["ship_id", "ship_type", "weather_conditions"] + feature_cols + ["reconstruction_error"]]
top5


---

## 9. Autoencoders vs. PCA, and beyond anomaly detection

`NB09` used PCA to compress the 60-dimensional Sonar data down to 2 components for visualization. An autoencoder's bottleneck does something structurally similar — compress, then reconstruct — but with one key difference: PCA is restricted to **linear** projections, while an autoencoder's `ReLU` layers let it learn **non-linear** compression. On data with genuinely non-linear structure, `an autoencoder can capture patterns PCA's straight-line projections cannot`; on data that's close to linear, the two often perform similarly, and PCA is far simpler to train and interpret.

Anomaly detection is only one application. The same compress-and-reconstruct idea also underlies:

| Application | How the bottleneck is used |
|---|---|
| Anomaly detection (today) | High reconstruction error flags unusual examples |
| Denoising | Train on noisy input, clean target; the bottleneck learns to discard noise |
| Dimensionality reduction | Use the bottleneck values themselves as compressed features for another model |
| Pretraining | Train an encoder unsupervised on abundant unlabeled data, then reuse it — conceptually related to `NB15`'s transfer learning, but the "pretraining" happens on your own unlabeled data instead of someone else's labeled ImageNet |

**Try it yourself**: test the linear-vs-non-linear claim directly. Fit a 2-component PCA on the same scaled data, reconstruct it (`pca.inverse_transform`), and compare its mean reconstruction error to the autoencoder's — does PCA really perform similarly here, or does the autoencoder's non-linearity actually help on this data?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_pca_reconstructed = pca.inverse_transform(X_pca)
pca_reconstruction_error = ((X_pca_reconstructed - X_scaled) ** 2).mean(axis=1)

print(f"Autoencoder mean reconstruction error: {reconstruction_error.mean():.4f}")
print(f"PCA (2 components) mean reconstruction error: {pca_reconstruction_error.mean():.4f}")


See which one wins on your run. If PCA comes out ahead (or very close), that's consistent with this data's relationships being close to linear — exactly the case this section's text describes, where PCA's simplicity is the better practical choice despite the autoencoder's extra flexibility.

---

## Class summary

- An autoencoder trains to reconstruct its own input through a narrow bottleneck — no labels, the input is its own target.
- The bottleneck forces compression; what the network fails to compress well (rare, unusual patterns) reconstructs poorly.
- Reconstruction error becomes an anomaly score with zero labeled anomalies needed — a real practical advantage over every supervised classifier in this block.
- We trained a real autoencoder on real ship fuel data, chose a threshold, flagged anomalous voyages, and checked whether the flags lined up with weather or ship type.
- Autoencoders generalize PCA's compression idea to non-linear relationships, and the same architecture underlies denoising, dimensionality reduction, and self-supervised pretraining.

## For the next class (NB17)

We close Block 3 with a review across all its architectures — MLP, CNN, RNN/LSTM, transfer learning, autoencoders — and a discussion of how to choose between them for a new, unseen problem.

## Homework / Practice Ideas

1. Change `bottleneck_size` from 2 to 1 and to 3 — how does the training reconstruction loss change? What does a bottleneck of 1 imply about how much the model can actually represent?
2. Change the anomaly threshold from the 95th to the 99th percentile — how many voyages get flagged now, and does the weather/ship-type crosstab from Part 8 tell a different story?
3. Add `CO2_emissions` back into `feature_cols` despite `NB07`'s leakage warning — does the autoencoder's reconstruction error change much? Given how correlated it is with `fuel_consumption`, would you expect it to?
4. Compare the voyages flagged here against the cluster assignments from `NB09` (if you still have that notebook's output) — do anomalies tend to fall in one particular cluster, or spread across all of them?
5. In your own words, explain why an autoencoder trained on mostly-normal data can still detect anomalies it never explicitly saw labeled as such — what exactly is it optimizing for, and why does that generalize to flagging the unusual?

> ***As always: an unsupervised flag is a starting point for investigation, not a verdict — the crosstab and scatter plot in Part 8 are the difference between "the model said so" and actually understanding what got flagged.***
